# Clase 6: Fine-Tuning Eficiente (QLoRA) y Alineación (SFT)
**Prof. Leticia Rodriguez | Universidad de Buenos Aires**

En esta libreta nos enfocaremos en la práctica del ajuste de parámetros eficiente (PEFT). Dejaremos la teoría de lado y veremos el impacto real en la infraestructura al entrenar un modelo de lenguaje.

Implementaremos un flujo robusto combinando dos metodologías críticas:
1. **QLoRA (Quantized Low-Rank Adaptation):** Compresión del modelo base a 4-bits e inyección de matrices adaptables para optimizar drásticamente la memoria VRAM.
2. **SFT (Supervised Fine-Tuning):** Entrenamiento supervisado con un dataset de instrucciones para alinear el comportamiento del modelo.

In [1]:
# Instalamos el ecosistema de Hugging Face necesario para PEFT, Cuantización y Entrenamiento
!pip install -q -U transformers peft accelerate bitsandbytes datasets huggingface_hub trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.4/684.4 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.1 MB/s eta 0:00:00


In [2]:
import os
import torch
from google.colab import userdata
from huggingface_hub import login

# Configuración de credenciales utilizando los secretos de Colab
hf_key = userdata.get('HF_TOKEN')
login(token=hf_key)
MODELO_BASE = 'google/gemma-2b'
print("Entorno de Fine-Tuning configurado e inicializado.")

Entorno de Fine-Tuning configurado e inicializado.


## PARTE 1: Carga del Modelo Base y Cuantización (La "Q" de QLoRA)
Para evitar el colapso de la memoria (Out of Memory) en nuestra GPU, utilizamos `BitsAndBytesConfig` para comprimir los pesos originales del modelo fundacional a 4-bits mientras se carga en la memoria de video.

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Configuración de Cuantización a 4-bits
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("--- DESCARGANDO Y COMPRIMIENDO MODELO BASE ---")
model = AutoModelForCausalLM.from_pretrained(
    MODELO_BASE,
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
print("¡Modelo cargado exitosamente en memoria!")

--- DESCARGANDO Y COMPRIMIENDO MODELO BASE ---


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/33.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

¡Modelo cargado exitosamente en memoria!


## PARTE 2: Preparación del Dataset de Instrucciones
Para que el modelo aprenda a responder como un asistente útil (SFT), necesitamos proveerle ejemplos estructurados. Cargaremos un subconjunto de datos ligero para acelerar la demostración en clase.

In [4]:
from datasets import load_dataset

print("--- OBTENIENDO DATOS DE ENTRENAMIENTO ---")
dataset = load_dataset("mlabonne/guanaco-llama2-1k", split="train")

print(f"Dataset cargado con éxito. Cantidad de filas: {len(dataset)}")
print("\n--- MUESTRA DEL PRIMER REGISTRO ---")
print(dataset[0]['text'][:300] + "...")

--- OBTENIENDO DATOS DE ENTRENAMIENTO ---


README.md:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

data/train-00000-of-00001-9ad84bb9cf65a4(…):   0%|          | 0.00/967k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset cargado con éxito. Cantidad de filas: 1000

--- MUESTRA DEL PRIMER REGISTRO ---
<s>[INST] Me gradué hace poco de la carrera de medicina ¿Me podrías aconsejar para conseguir rápidamente un puesto de trabajo? [/INST] Esto vale tanto para médicos como para cualquier otra profesión tras finalizar los estudios aniversarios y mi consejo sería preguntar a cuántas personas haya conocid...


In [5]:
print(dataset[0])

{'text': '<s>[INST] Me gradué hace poco de la carrera de medicina ¿Me podrías aconsejar para conseguir rápidamente un puesto de trabajo? [/INST] Esto vale tanto para médicos como para cualquier otra profesión tras finalizar los estudios aniversarios y mi consejo sería preguntar a cuántas personas haya conocido mejor. En este caso, mi primera opción sería hablar con otros profesionales médicos, echar currículos en hospitales y cualquier centro de salud. En paralelo, trabajaría por mejorar mi marca personal como médico mediante un blog o formas digitales de comunicación como los vídeos. Y, para mejorar las posibilidades de encontrar trabajo, también participaría en congresos y encuentros para conseguir más contactos. Y, además de todo lo anterior, seguiría estudiando para presentarme a las oposiciones y ejercer la medicina en el sector público de mi país. </s>'}


## PARTE 3: Inyección de Adaptadores LoRA y Entrenamiento
Aislamos los pesos originales congelados y agregamos nuestras matrices de bajo rango. Finalmente, orquestamos el entrenamiento utilizando el `SFTTrainer`. Limitaremos los pasos de entrenamiento para que la ejecución sea rápida durante la lección.

In [ ]:
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig # <--- Importamos la configuración de los adaptadores

# 1. Configuramos los adaptadores LoRA
peft_config = LoraConfig(
    r=16,               # El "rango" de las matrices. 8 o 16 es ideal para empezar sin saturar la VRAM.
    lora_alpha=32,      # Escala del adaptador (suele ser el doble de 'r').
    lora_dropout=0.05,  # Evita el sobreajuste (overfitting).
    bias="none",        # Recomendado para LoRA.
    task_type="CAUSAL_LM" # Tipo de tarea: Modelado de Lenguaje Causal.
)

# 2. Configuración de entrenamiento (la que ya tenías)
training_args = SFTConfig(
    output_dir="./resultados",
    dataset_text_field="text",
    max_length=1024,
    learning_rate=2e-4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,      # 4. LA MAGIA: Intercambia un poco de CPU por mucha VRAM
    optim="paged_adamw_8bit",         # 5. Optimizador en 8-bit para reducir el peso de los estados

)



# 3. Ensamblamos el SFTTrainer inyectando el peft_config
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    peft_config=peft_config
)

# 4. Iniciar el entrenamiento
trainer.train()

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Step,Training Loss
10,1.893526
20,1.720194
30,1.716640
40,1.688308
50,1.741409
60,1.639646
70,1.673390
80,1.829192
90,1.903521
100,1.657418


KeyboardInterrupt: 